# PoC — Speech-to-Text (STT) para carga de transacciones por voz

**Spike STT — TecaTrack**

Este notebook compara alternativas de STT basadas en **Whisper (modelo abierto)** para
transcribir audios cortos en español (es-AR) donde el usuario dicta una transacción.

Se evalúan dos ejes:

1. **Self-hosted** con `faster-whisper` (backend CTranslate2, CPU, int8): `small` y `large-v3-turbo`.
2. **API gestionada** (opcional, celdas al final): Groq / Hugging Face sirviendo Whisper.

El caso de uso es **carga de transacciones por voz**, procesada de forma **asíncrona**
respecto a la respuesta del backend (la latencia no es un requisito duro).

> Lo que más importa para el negocio es **acertar el monto** (sin importar si se escribe
> en letras o en dígitos), y secundariamente las **entidades** (banco origen/destino).


## 1. Datos

- **Audios reales**: 3 grabaciones provistas (`audios/transaccion*.wav`, ~11–16 s, 44.1 kHz).
- **Audios sintéticos**: generados con `edge-tts` (voces es-AR), con texto controlado
  para tener *ground truth* exacto y variar montos/entidades.


In [1]:
import os, time, unicodedata, re, asyncio
from pathlib import Path

BASE = Path("audios")
SYNTH_DIR = BASE / "synthetic"
SYNTH_DIR.mkdir(parents=True, exist_ok=True)

# Modelos self-hosted a evaluar (faster-whisper, CPU int8)
MODELS = ["small", "large-v3-turbo"]
DEVICE = "cpu"
COMPUTE = "int8"
LANG = "es"


In [2]:
# --- Audios reales + ground truth ---
# 'amount' = valor numerico correcto; 'entities' = bancos esperados.
REAL = [
    {
        "file": BASE / "transaccion1.wav",
        "text": "Mande veinte mil doscientos treinta y ocho pesos desde tecabank a mi cuenta de brubank",
        "amount": 20238,
        "entities": ["tecabank", "brubank"],
    },
    {
        "file": BASE / "transaccion2.wav",
        "text": "Gaste quinientos pesos en una cosa que compre con mi cuenta de brubank",
        "amount": 500,
        "entities": ["brubank"],
    },
    {
        "file": BASE / "transaccion3.wav",
        "text": "Desde la cuenta de lemon pase ochocientos cincuenta y dos pesos a un amigo",
        "amount": 852,
        "entities": ["lemon"],
    },
]


### Generación de audios sintéticos (es-AR)

`edge-tts` no requiere API key. Si no hay conexión, esta celda se omite y la evaluación
continúa solo con los audios reales.

In [3]:
import edge_tts

SYNTH_SPECS = [
    {"voice": "es-AR-ElenaNeural",
     "text": "Transferi quince mil pesos a la cuenta de Lemon",
     "amount": 15000, "entities": ["lemon"]},
    {"voice": "es-AR-TomasNeural",
     "text": "Pague tres mil quinientos cuarenta pesos con Brubank",
     "amount": 3540, "entities": ["brubank"]},
    {"voice": "es-AR-ElenaNeural",
     "text": "Recibi cien mil doscientos pesos en mi cuenta de Tecabank",
     "amount": 100200, "entities": ["tecabank"]},
    {"voice": "es-AR-TomasNeural",
     "text": "Gaste siete mil ochocientos noventa y nueve pesos en el supermercado",
     "amount": 7899, "entities": []},
]

async def _tts(text, voice, out):
    await edge_tts.Communicate(text, voice).save(str(out))

def run_async(coro):
    "Ejecuta una corrutina tanto en script como dentro de un loop (Jupyter)."
    import threading
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    box = {}
    def _r():
        box["v"] = asyncio.run(coro)
    t = threading.Thread(target=_r); t.start(); t.join()
    return box.get("v")

SYNTH = []
try:
    for i, s in enumerate(SYNTH_SPECS, 1):
        out = SYNTH_DIR / f"synth{i}.mp3"
        run_async(_tts(s["text"], s["voice"], out))
        SYNTH.append({"file": out, "text": s["text"],
                      "amount": s["amount"], "entities": s["entities"]})
    print(f"Generados {len(SYNTH)} audios sinteticos")
except Exception as e:
    print("TTS no disponible (se sigue solo con audios reales):", e)


Generados 4 audios sinteticos


## 2. Métricas

Se reportan **tres** métricas, no solo WER, para reflejar lo que importa al negocio:

- **Amount OK**: el valor numérico correcto aparece en la transcripción, sin importar si
  se escribe en letras o dígitos. Un número **partido** (`20 238`) cuenta como *error*.
- **WER normalizado**: se pasa todo a minúsculas, se quitan acentos y muletillas (`emm`, `eh`),
  y se canonizan números (letras↔dígitos) antes de comparar. Mide calidad de transcripción
  de forma justa.
- **Entities OK**: fracción de bancos esperados detectados en la transcripción.


In [4]:
import jiwer

UNITS = {"cero":0,"un":1,"uno":1,"una":1,"dos":2,"tres":3,"cuatro":4,"cinco":5,
 "seis":6,"siete":7,"ocho":8,"nueve":9,"diez":10,"once":11,"doce":12,"trece":13,
 "catorce":14,"quince":15,"dieciseis":16,"diecisiete":17,"dieciocho":18,
 "diecinueve":19,"veinte":20,"veintiuno":21,"veintidos":22,"veintitres":23,
 "veinticuatro":24,"veinticinco":25,"veintiseis":26,"veintisiete":27,
 "veintiocho":28,"veintinueve":29,"treinta":30,"cuarenta":40,"cincuenta":50,
 "sesenta":60,"setenta":70,"ochenta":80,"noventa":90,"cien":100,"ciento":100,
 "doscientos":200,"trescientos":300,"cuatrocientos":400,"quinientos":500,
 "seiscientos":600,"setecientos":700,"ochocientos":800,"novecientos":900}

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn")

SCALES = {"mil": 1000, "millon": 1_000_000, "millones": 1_000_000}
NUMWORD = set(UNITS) | set(SCALES)

def _is_numtok(t):
    return t.isdigit() or t in NUMWORD

def _parse_run(tokens):
    "Parsea una secuencia de tokens numericos (digitos y/o palabras) a int."
    total, current = 0, 0
    for t in tokens:
        if t.isdigit():
            current += int(t)
        elif t in UNITS:
            current += UNITS[t]
        elif t in SCALES:
            current = (current or 1) * SCALES[t]
            total += current
            current = 0
    return total + current

def words_to_numbers(tokens):
    "Convierte numeros en palabras y/o digitos (ej '15 mil', 'veinte mil ocho') a int."
    out, i, n = [], 0, len(tokens)
    while i < n:
        if _is_numtok(tokens[i]):
            run = []
            while i < n and (_is_numtok(tokens[i]) or
                  (tokens[i] == "y" and run and i + 1 < n and _is_numtok(tokens[i + 1]))):
                if tokens[i] != "y":
                    run.append(tokens[i])
                i += 1
            out.append(str(_parse_run(run)))
        else:
            out.append(tokens[i]); i += 1
    return out

def normalize(text):
    t = strip_accents(text.lower())
    t = re.sub(r"[.,](?=\d{3}\b)", "", t)          # separador de miles 20.238 -> 20238
    t = re.sub(r"\b(e+m+|e+h+|m+h+|este)\b", " ", t)  # muletillas
    t = re.sub(r"[^\w\s]", " ", t)
    toks = words_to_numbers(t.split())
    return " ".join(toks).strip()

def norm_wer(ref, hyp):
    return jiwer.wer(normalize(ref), normalize(hyp))

def amounts_in(text):
    "Conjunto de enteros presentes (digitos contiguos o palabras)."
    n = normalize(text)
    return {int(x) for x in re.findall(r"\b\d+\b", n)}

def split_number_present(raw, value):
    "True si el valor aparece partido por espacios (ej '20 238')."
    digits = str(value)
    if len(digits) <= 3:
        return False
    spaced = re.sub(r"(\d)(?=(\d{3})+\b)", r"\1 ", digits)  # '20 238'
    # Solo cuenta como error un espacio literal entre grupos; el punto de miles
    # (formato es-AR '20.238') es valido y NO se trata como numero partido.
    return spaced in raw

def amount_ok(raw, value):
    if split_number_present(raw, value):
        return False
    return value in amounts_in(raw)

def entities_ok(raw, ents):
    if not ents:
        return None
    n = strip_accents(raw.lower())
    hits = sum(1 for e in ents if strip_accents(e.lower()) in n)
    return hits / len(ents)


## 3. Transcripción y evaluación

Se cargan los modelos una sola vez y se transcriben todos los audios (reales + sintéticos).

In [5]:
from faster_whisper import WhisperModel

DATASET = [("real", d) for d in REAL] + [("synth", d) for d in SYNTH]

def transcribe(model, path):
    t = time.time()
    segs, _ = model.transcribe(str(path), language=LANG)
    txt = "".join(s.text for s in segs).strip()
    return txt, time.time() - t

results = []
for name in MODELS:
    t0 = time.time()
    model = WhisperModel(name, device=DEVICE, compute_type=COMPUTE)
    load = time.time() - t0
    print(f"\n=== {name} (load {load:.1f}s) ===")
    for kind, d in DATASET:
        hyp, dt = transcribe(model, d["file"])
        results.append({
            "model": name, "kind": kind, "file": d["file"].name,
            "ref": d["text"], "hyp": hyp,
            "amount_ok": amount_ok(hyp, d["amount"]),
            "wer": norm_wer(d["text"], hyp),
            "ent_ok": entities_ok(hyp, d["entities"]),
            "secs": dt,
        })
        print(f"[{kind}] {d['file'].name}: amount={'OK' if results[-1]['amount_ok'] else 'X'} "
              f"wer={results[-1]['wer']:.2f} ({dt:.1f}s)")
        print("   ->", hyp)
    del model



=== small (load 1.6s) ===


[real] transaccion1.wav: amount=OK wer=0.10 (1.9s)
   -> Mandé 20.238 pesos desde Tecabank a mi cuenta de Bluebank.


[real] transaccion2.wav: amount=OK wer=0.15 (1.8s)
   -> Las té 500 pesos en una cosa que compré con mi cuenta de Brubank.


[real] transaccion3.wav: amount=OK wer=0.00 (1.8s)
   -> Desde la cuenta de Lemon, pasé 852 pesos a un amigo.


[synth] synth1.mp3: amount=OK wer=0.12 (1.7s)
   -> Transferí 15 mil pesos a la cuenta de Lehman.


...

[real] transaccion3.wav: amount=OK wer=0.00 (10.0s)
   -> Desde la cuenta de Lemon pasé 852 pesos a un amigo


[synth] synth1.mp3: amount=OK wer=0.25 (10.4s)
   -> Transferí 15.000 pesos a la cuenta del Emen.


[synth] synth2.mp3: amount=OK wer=0.00 (11.4s)
   -> Pague 3.540 pesos con Brubank.


[synth] synth3.mp3: amount=OK wer=0.00 (10.0s)
   -> Recibí 100.200 pesos en mi cuenta de Tecabank.


[synth] synth4.mp3: amount=OK wer=0.00 (10.0s)
   -> Gaste 7,899 pesos en el supermercado.


In [6]:
from tabulate import tabulate

rows = []
for name in MODELS:
    r = [x for x in results if x["model"] == name]
    amt = sum(1 for x in r if x["amount_ok"]) / len(r)
    wer = sum(x["wer"] for x in r) / len(r)
    ents = [x["ent_ok"] for x in r if x["ent_ok"] is not None]
    ent = sum(ents) / len(ents) if ents else float("nan")
    secs = sum(x["secs"] for x in r) / len(r)
    rows.append([name, f"{amt:.0%}", f"{wer:.2f}", f"{ent:.0%}", f"{secs:.1f}s"])

print(tabulate(rows,
    headers=["Modelo", "Amount OK", "WER norm.", "Entities OK", "Tiempo/clip"],
    tablefmt="github"))


| Modelo         | Amount OK   |   WER norm. | Entities OK   | Tiempo/clip   |
|----------------|-------------|-------------|---------------|---------------|
| small          | 100%        |        0.07 | 75%           | 1.8s          |
| large-v3-turbo | 100%        |        0.06 | 58%           | 9.8s          |


## 4. API gestionada (opcional)

Estas celdas demuestran la misma transcripción contra una **API gestionada** que sirve
Whisper (modelo abierto) con **cuota gratuita**, para no consumir RAM local. Solo corren
si está definida la variable de entorno correspondiente; de lo contrario se omiten.

- **Groq** (`GROQ_API_KEY`): Whisper `large-v3` / `large-v3-turbo`, free tier.
- **Hugging Face** (`HF_TOKEN`): Inference Providers, crédito mensual gratuito.

In [7]:
# --- Groq (opcional) ---
if os.getenv("GROQ_API_KEY"):
    from groq import Groq
    client = Groq()
    for kind, d in DATASET:
        with open(d["file"], "rb") as f:
            r = client.audio.transcriptions.create(
                file=(d["file"].name, f.read()),
                model="whisper-large-v3-turbo", language="es")
        print(f"[groq {kind}] {d['file'].name}: amount="
              f"{'OK' if amount_ok(r.text, d['amount']) else 'X'}")
        print("   ->", r.text.strip())
else:
    print("GROQ_API_KEY no definido — celda omitida.")


GROQ_API_KEY no definido — celda omitida.


In [8]:
# --- Hugging Face Inference Providers (opcional) ---
if os.getenv("HF_TOKEN"):
    from huggingface_hub import InferenceClient
    client = InferenceClient(api_key=os.environ["HF_TOKEN"])
    for kind, d in DATASET:
        out = client.automatic_speech_recognition(
            str(d["file"]), model="openai/whisper-large-v3")
        txt = out.text if hasattr(out, "text") else str(out)
        print(f"[hf {kind}] {d['file'].name}: amount="
              f"{'OK' if amount_ok(txt, d['amount']) else 'X'}")
        print("   ->", txt.strip())
else:
    print("HF_TOKEN no definido — celda omitida.")


HF_TOKEN no definido — celda omitida.


## 5. Conclusión

Ver el informe comparativo (`stt-spike-report.md`) para la recomendación fundamentada
a partir de estos resultados.